
# How to Save the Pipeline Memory Efficiently

Our pipeline offers extensive insights into the dynamics of training and other intermediate results.  
When re-using the pipeline (with `.save` and `.load` functionality), this can create very large binary files, especially when working with large datasets.  

Therefore, the default way of saving the pipeline is by discarding most of the training dynamics and intermediate results.  
In this tutorial, we show:  
- Which data is kept and which is discarded. <br><br>
- The trade-offs in visualizing and evaluating a saved pipeline. <br><br>
- How to save the complete pipeline including all training dynamics and intermediate results. <br><br>

## Theory: What Data is Kept?

### Result Object
The main driver of memory is the `result` attribute and the `_datasets` (depending on the data size) in the pipeline and the trainer. 

In the `result` attribute, we save the following:  
- model
- adata_latent <br><br>
All other attributes are set to empty default values.
### Pipeline
Here we keep all attributes excpept:
- `.visualizer`
- `_datasets`
We remove the  `.visualizer`, because this stores plots which can be misleading when rerunning the pipeline and seeing old plots that do not fit the new data.    
We remove the `_datasets` attribute, because it is often uses the most memory.
All other attributes are kept. This is especially relevant for the `config` and the `_preprocessor`.  
Whenever you perform any pipeline step, it will use the initial config you passed when first creating the pipeline.  
The preprocessor is necessary because when running `predict` with new data, this should be preprocessed the same way as initially (e.g., which genes to keep, how the scalers were fitted, etc.).

### Trainer
We remove most attribute of the `_trainer` via the method `purge`.

**IMPORTANT**
> When calling `.save`, the `result` attribute is modified in-place (to avoid copies).  
> This means after calling `.save`, your current pipeline object does not contain information from the other result objects.  
> This applies to the pipeline object in current memory, not only to the pipeline object after loading.  
> If you want to avoid this, please run `.save(save_all=True)`.


## Practical Examples
### First we run a varix pipeline 

In [ ]:
from autoencodix.utils.example_data import EXAMPLE_MULTI_SC
from autoencodix.configs.varix_config import VarixConfig
from autoencodix.configs.default_config import DataCase, DataConfig, DataInfo
import autoencodix as acx

my_config = VarixConfig(
    learning_rate=0.001,
    epochs=30,
    checkpoint_interval=5,
    default_vae_loss="kl",  # kl or mmd possible
    data_case=DataCase.MULTI_SINGLE_CELL,
)
print("\n")
print("Starting Pipeline")
print("-" * 50)
print("-" * 50)
varix = acx.Varix(
    data=EXAMPLE_MULTI_SC,
    config=my_config,
)
result = varix.run()

#### Saving and Loading Explained

Here we first check the latent space (could be any other training dynamic) before saving.  
After saving, we see that the `result` object got cleaned and only `model`, `adata_latent`, and `embedding_evaluation` are kept.


In [ ]:
ls_before_save = result.latentspaces.get(epoch=-1, split="test")
print("Length of latentspace before saving")
print(len(ls_before_save))
varix.save("varix_backup.pkl")
ls_after_save = result.latentspaces.get()
print("Length of latentspace after saving")
print(len(ls_after_save))

#### Implications for Visualization
We load the model and see the implications of this in `visualize`.


In [ ]:
varix_loaded = acx.Varix().load("varix_backup.pkl")

In [ ]:
varix_loaded.visualize()

> Note that we code a `UserWarning`. This is expected, since it tells us that we cannot visualize loss plots anymore because the data for this is no longer in the result object.  
> However, we could run a predict step on the trained model to get some visualizations.


In [ ]:
res_loaded = varix_loaded.predict(data=EXAMPLE_MULTI_SC)

In [ ]:
varix_loaded.visualize()
# we use test split here, because predict with new data is equal to test split
varix_loaded.show_result(split="test")

#### Implications for Evaluate
The evaluate step is not supported for this memory-efficient saving as of now, and you'll get the following warning and error:


In [ ]:
res_eval = varix_loaded.evaluate(params=["cell_type"])

## How to Keep All Data
When you really want to investigate all training dynamics and results, we recommend setting `save_all=True`. See below:


In [ ]:
from autoencodix.utils.example_data import EXAMPLE_MULTI_BULK

my_config = VarixConfig(
    learning_rate=0.001,
    epochs=30,
    checkpoint_interval=5,
    default_vae_loss="kl",  # kl or mmd possible
    data_case=DataCase.MULTI_BULK,
)
n_varix = acx.Varix(data=EXAMPLE_MULTI_BULK, config=my_config)
result_new = n_varix.run()
n_varix.save(file_path="new_varix.pkl", save_all=True)

Now we can load the pipeline and all intermediate data is kept.

In [ ]:
loaded_all_varix = acx.Varix().load(file_path="new_varix.pkl")

In [ ]:
loaded_all_varix.visualize()

In [ ]:
loaded_all_varix.show_result()

And we see that the loss plots from the initial training are still present.

## Implications for Ontix and XModalix
These two pipelines offer more/other visualizations.
### ❗❗ Requirements: Getting Tutorial Data for XModalixj ❗❗

You can use the following bash commands to download the data and set up the correct folder structure.  
**Assumption:** you are in the root of `autoencodix_package`.

```bash
mkdir -p data
cd data
wget "https://cloud.scadsai.uni-leipzig.de/index.php/s/bq64MaQyZGZfN64/download/XModalix-Tut-data.zip"
unzip XModalix-Tut-data.zip
```


### XModalix

In [ ]:
import os
import autoencodix as acx

from autoencodix.configs.xmodalix_config import XModalixConfig
from autoencodix.configs.default_config import DataCase

p = os.getcwd()
d = "autoencodix_package"
if d not in p:
    raise FileNotFoundError(f"'{d}' not found in path: {p}")
os.chdir(os.sep.join(p.split(os.sep)[: p.split(os.sep).index(d) + 1]))
print(f"Changed to: {os.getcwd()}")

clin_file = os.path.join("data/XModalix-Tut-data/combined_clin_formatted.parquet")
rna_file = os.path.join("data/XModalix-Tut-data/combined_rnaseq_formatted.parquet")
img_root = os.path.join("data/XModalix-Tut-data/images/tcga_fake")

xmodalix_config = XModalixConfig(
    checkpoint_interval=5,
    class_param="CANCER_TYPE_ACRONYM",
    epochs=30,
    latent_dim=8,
    requires_paired=False,
    pretrain_epochs=2,
    data_case=DataCase.IMG_TO_BULK,
    data_config=DataConfig(
        annotation_columns=["CANCER_TYPE_ACRONYM"],
        data_info={
            "img": DataInfo(
                file_path=img_root,
                data_type="IMG",
                scaling="MINMAX",
                translate_direction="to",
                pretrain_epochs=2,
                # extra_anno_file=imganno_file,
            ),
            "rna": DataInfo(
                file_path=rna_file,
                data_type="NUMERIC",
                scaling="MINMAX",
                translate_direction="from",
            ),
            "anno": DataInfo(file_path=clin_file, data_type="ANNOTATION", sep="\t"),
        },
    ),
)

xmodalix = acx.XModalix(config=xmodalix_config)
result = xmodalix.run()
outpath = os.path.join("tutorial_res", "xmodalix.pkl")
xmodalix.save(file_path=outpath)



Now we can load the pipeline again

In [ ]:
xmodalix_loaded = acx.XModalix.load(outpath)

Since we used memory efficient saving, we did not save the data, thus we need to run preprocessing again, this will read and preprocess the files defined in the config.

In [ ]:
xmodalix_loaded.preprocess()


No we can use the trained model to predict again:

In [ ]:
# now you can use the model to predict with a different pair again:

r = xmodalix_loaded.predict(from_key="rna", to_key="img")

In [ ]:
xmodalix_loaded.visualize()

In [ ]:
xmodalix_loaded.show_result()

### Ontix

In [ ]:
import autoencodix as acx
from autoencodix.configs.default_config import DefaultConfig
from autoencodix.configs.ontix_config import OntixConfig

from autoencodix.utils.example_data import (
    EXAMPLE_PROCESSED_DATA,
)

# EXAMPLE_DATA hold PyTorch Datasets (child with extra info) with metdata for train, test and valid splits
processed_data = EXAMPLE_PROCESSED_DATA


ont_lvl1 = dict()
ont_lvl2 = dict()

ont_lvl1["pwy-1"] = ["sub-pwy-1", "sub-pwy-2"]
ont_lvl1["pwy-2"] = ["sub-pwy-2"]
ont_lvl1["pwy-3"] = ["sub-pwy-1", "sub-pwy-3"]
# first third of feature ids in processed_data.train.feature_ids
ont_lvl2["sub-pwy-1"] = processed_data.train.feature_ids[
    : int(len(processed_data.train.feature_ids) / 3)
]
# second third of feature ids in processed_data.train.feature_ids
ont_lvl2["sub-pwy-2"] = processed_data.train.feature_ids[
    int(len(processed_data.train.feature_ids) / 3) : int(
        2 * len(processed_data.train.feature_ids) / 3
    )
]
# last third of feature ids in processed_data.train.feature_ids
ont_lvl2["sub-pwy-3"] = processed_data.train.feature_ids[
    int(2 * len(processed_data.train.feature_ids) / 3) : int(
        len(processed_data.train.feature_ids)
    )
]

# ont_lvl2["sub-pwy-1"] = ["gene-1", "gene-2"]
# ont_lvl2["sub-pwy-2"] = ["gene-3", "gene-4"]
# ont_lvl2["sub-pwy-3"] = ["gene-2", "gene-6"]

ontology_tuple = (ont_lvl1, ont_lvl2)

# Write each dictionary in ontology_tuple to a separate text file
for i, ont_dict in enumerate(ontology_tuple):
    file_name = f"ontology_level_{i + 1}.txt"
    with open(file_name, "w") as f:
        for key, values in ont_dict.items():
            for value in values:
                f.write(f"{value}\t{key}\n")
print("Ontology dictionaries written to ontology_level_1.txt and ontology_level_2.txt")

ont_files = ["ontology_level_1.txt", "ontology_level_2.txt"]

In [ ]:
ontix = acx.Ontix(
    ontologies=ont_files,
    sep="\t",
    config=OntixConfig(epochs=30, learning_rate=0.005, n_layers=1),
    data=processed_data,
)
result_onitx = ontix.run()

In [ ]:
ontix.save(file_path="ontix_backup.pkl")


In [ ]:
ontix_loaded = acx.Ontix(ontologies=ont_files).load(file_path="ontix_backup.pkl")

In [ ]:
pred_res = ontix_loaded.predict(EXAMPLE_PROCESSED_DATA)

In [ ]:
ontix_loaded.visualize()

ontix_loaded.evaluate(params=["cluster"])

In [ ]:
ontix_loaded.show_result()